# 🌫️ Air Quality TinyML Model for ESP32
### Trained on Smoke Detection Dataset | Kaggle Notebook
**Output:** A TensorFlow Lite model ready to deploy on ESP32

## Step 1 — Install & Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

import os

print('TensorFlow version:', tf.__version__)

## Step 2 — Load Dataset

In [ ]:
# Load the smoke detection dataset from Kaggle
# Dataset: 'smoke-detection-dataset' on Kaggle
df = pd.read_csv('/kaggle/input/smoke-detection-dataset/smoke_detection_iot.csv')

print('Dataset shape:', df.shape)
print('\nColumns:', df.columns.tolist())
print('\nFirst 5 rows:')
df.head()

## Step 3 — Select Only Relevant Columns (Your Sensors)

In [ ]:
# Keep only columns your ESP32 sensors can measure:
# Temperature  → DHT11
# Humidity     → DHT11
# Raw Ethanol  → MQ2 (closest match)
# Fire Alarm   → Label (what we want to predict)

df_model = df[['Temperature[C]', 'Humidity[%]', 'Raw Ethanol', 'Fire Alarm']].copy()

# Rename for cleaner code
df_model.columns = ['temperature', 'humidity', 'air_value', 'label']

print('Selected dataset shape:', df_model.shape)
print('\nLabel distribution:')
print(df_model['label'].value_counts())
print('\nMissing values:')
print(df_model.isnull().sum())

## Step 4 — Explore & Visualize Data

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Temperature distribution
axes[0].hist(df_model[df_model['label']==0]['temperature'], alpha=0.7, label='Safe', color='green', bins=30)
axes[0].hist(df_model[df_model['label']==1]['temperature'], alpha=0.7, label='Fire/Smoke', color='red', bins=30)
axes[0].set_title('Temperature Distribution')
axes[0].set_xlabel('Temperature (°C)')
axes[0].legend()

# Humidity distribution
axes[1].hist(df_model[df_model['label']==0]['humidity'], alpha=0.7, label='Safe', color='green', bins=30)
axes[1].hist(df_model[df_model['label']==1]['humidity'], alpha=0.7, label='Fire/Smoke', color='red', bins=30)
axes[1].set_title('Humidity Distribution')
axes[1].set_xlabel('Humidity (%)')
axes[1].legend()

# Air value distribution
axes[2].hist(df_model[df_model['label']==0]['air_value'], alpha=0.7, label='Safe', color='green', bins=30)
axes[2].hist(df_model[df_model['label']==1]['air_value'], alpha=0.7, label='Fire/Smoke', color='red', bins=30)
axes[2].set_title('Air Value (Raw Ethanol) Distribution')
axes[2].set_xlabel('Raw Ethanol')
axes[2].legend()

plt.tight_layout()
plt.show()

## Step 5 — Prepare Features & Labels

In [ ]:
# Drop any rows with missing values
df_model = df_model.dropna()

# Features (X) and Labels (y)
X = df_model[['temperature', 'humidity', 'air_value']].values
y = df_model['label'].values

# Split into train and test sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train size:', X_train.shape)
print('Test size:', X_test.shape)

## Step 6 — Normalize the Data

In [ ]:
# Normalize so all features are on the same scale
# IMPORTANT: Save mean and std — you need these in ESP32 code too!
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Print normalization values — copy these into your ESP32 code!
print('=== COPY THESE INTO YOUR ESP32 CODE ===')
print('Mean values:  ', scaler.mean_)
print('Scale values: ', scaler.scale_)
print('========================================')

## Step 7 — Build Lightweight Neural Network

In [ ]:
# Small model — designed to fit on ESP32 memory
model = keras.Sequential([
    layers.Input(shape=(3,)),           # 3 inputs: temp, humidity, air
    layers.Dense(16, activation='relu'),# Hidden layer 1 — small
    layers.Dense(8,  activation='relu'),# Hidden layer 2 — smaller
    layers.Dense(1,  activation='sigmoid') # Output: 0=Safe, 1=Fire/Smoke
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

## Step 8 — Train the Model

In [ ]:
# Early stopping — stops if model stops improving
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train_scaled, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

## Step 9 — Plot Training Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Accuracy plot
axes[0].plot(history.history['accuracy'],     label='Train Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

# Loss plot
axes[1].plot(history.history['loss'],     label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Val Loss')
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.show()

## Step 10 — Evaluate the Model

In [ ]:
# Test accuracy
test_loss, test_acc = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f'Test Accuracy: {test_acc * 100:.2f}%')
print(f'Test Loss:     {test_loss:.4f}')

# Detailed report
y_pred = (model.predict(X_test_scaled) > 0.5).astype(int)
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Safe', 'Fire/Smoke']))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
            xticklabels=['Safe', 'Fire/Smoke'],
            yticklabels=['Safe', 'Fire/Smoke'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

## Step 11 — Convert to TensorFlow Lite (for ESP32)

In [ ]:
# Convert model to TensorFlow Lite with quantization
# Quantization makes the model smaller and faster on ESP32

converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Apply full integer quantization (smallest size for ESP32)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

# Save the .tflite file
tflite_path = 'air_quality_model.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

print(f'TFLite model saved: {tflite_path}')
print(f'Model size: {os.path.getsize(tflite_path)} bytes ({os.path.getsize(tflite_path)/1024:.2f} KB)')

## Step 12 — Convert to C Header File (for ESP32 Arduino)

In [ ]:
# Convert .tflite binary to a C array
# This is what you paste into your Arduino/ESP32 project

def convert_to_c_array(tflite_path, output_path='model_data.h'):
    with open(tflite_path, 'rb') as f:
        data = f.read()

    c_array = ', '.join([f'0x{b:02x}' for b in data])
    c_code = f'''// Auto-generated TFLite model for ESP32
// Air Quality Monitor - Fire/Smoke Detection
// Input:  temperature, humidity, air_value (normalized)
// Output: 0 = Safe, 1 = Fire/Smoke Detected

#ifndef MODEL_DATA_H
#define MODEL_DATA_H

const unsigned char model_data[] = {{
  {c_array}
}};
const unsigned int model_data_len = {len(data)};

// Normalization values from training — used in ESP32 code
// temperature_normalized = (temperature - TEMP_MEAN) / TEMP_SCALE
// humidity_normalized    = (humidity    - HUM_MEAN)  / HUM_SCALE
// air_normalized         = (air_value   - AIR_MEAN)  / AIR_SCALE

#endif
'''
    with open(output_path, 'w') as f:
        f.write(c_code)

    print(f'C header file saved: {output_path}')
    print(f'Array length: {len(data)} bytes')

convert_to_c_array('air_quality_model.tflite', 'model_data.h')

print('\n✅ Done! Download these two files:')
print('   1. air_quality_model.tflite')
print('   2. model_data.h  ← paste this into your Arduino project')

## Step 13 — Test Model with Sample ESP32 Values

In [ ]:
# Simulate what your ESP32 would send
test_cases = [
    {'temperature': 28.0, 'humidity': 55.0, 'air_value': 150,  'expected': 'Safe'},
    {'temperature': 45.0, 'humidity': 30.0, 'air_value': 800,  'expected': 'Fire/Smoke'},
    {'temperature': 32.0, 'humidity': 70.0, 'air_value': 300,  'expected': 'Moderate'},
    {'temperature': 55.0, 'humidity': 20.0, 'air_value': 1200, 'expected': 'Fire/Smoke'},
]

print('=== Sample Predictions ===')
for case in test_cases:
    raw = np.array([[case['temperature'], case['humidity'], case['air_value']]])
    scaled = scaler.transform(raw)
    pred = model.predict(scaled, verbose=0)[0][0]
    result = 'Fire/Smoke Detected 🔴' if pred > 0.5 else 'Safe ✅'
    print(f"Temp={case['temperature']}°C  Hum={case['humidity']}%  Air={case['air_value']}  → {result}  (confidence: {pred:.2f})")

## ✅ Summary

| Step | Done |
|------|------|
| Load & clean dataset | ✅ |
| Select sensor-matching columns | ✅ |
| Train lightweight neural network | ✅ |
| Evaluate accuracy | ✅ |
| Convert to TensorFlow Lite | ✅ |
| Convert to C header for ESP32 | ✅ |

### 📥 Download & Next Steps
1. Download `model_data.h` from Kaggle output
2. Place it in your Arduino project folder
3. Add TensorFlow Lite library to Arduino IDE
4. Use the mean & scale values printed in Step 6 in your ESP32 code
5. ESP32 reads sensors → normalizes → runs model → shows prediction on LCD + app